# Food Delivery Platform Analytics Dashboard
This notebook acts as the visualization layer for the SQL analytical queries executed against the `food_delivery.db` SQLite database. It visualizes customer behaviour, operations, partner logistics, and restaurant financials using `Pandas`, `Matplotlib`, and `Seaborn`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup database connection
conn = sqlite3.connect('../food_delivery.db')

# Setup consistent styling
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 7)
colors = ['#FF6F61', '#6B5B95', '#88B04B', '#F7CAC9', '#92A8D1', '#955251', '#B565A7', '#009B77', '#DD4124', '#D65076']
print("Connected to SQLite DB successfully!")

### Chart 1: Monthly Revenue Trend (GMV)
Demonstrates platform sales metrics over the chronological duration of the dataset.

In [ ]:
df1 = pd.read_sql_query('''
    SELECT strftime('%Y-%m', order_date) AS order_month, SUM(total_amount) AS revenue
    FROM orders WHERE order_status = 'Delivered'
    GROUP BY order_month ORDER BY order_month;
''', conn)

plt.figure(figsize=(12, 7))
plt.plot(df1['order_month'], df1['revenue'] / 1e6, marker='o', color='#FF6F61', linewidth=3, markersize=8)
plt.fill_between(df1['order_month'], df1['revenue'] / 1e6, color='#FF6F61', alpha=0.1)

peak_idx = df1['revenue'].idxmax()
plt.annotate(f"Peak: INR {df1.loc[peak_idx, 'revenue']/1e6:.2f}M ({df1.loc[peak_idx, 'order_month']})", 
             xy=(peak_idx, df1.loc[peak_idx, 'revenue']/1e6), 
             xytext=(peak_idx - 3, (df1.loc[peak_idx, 'revenue']/1e6) - 2),
             arrowprops=dict(facecolor='#333333', arrowstyle="->", connectionstyle="arc3,rad=-0.2"),
             fontweight='bold')

plt.title("Monthly Revenue Trend (GMV)", fontsize=18, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Revenue (in Millions INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Chart 2: City-wise Order Volume
Plots order aggregates across geographical locations.

In [ ]:
df2 = pd.read_sql_query('''
    SELECT city, COUNT(order_id) AS order_count FROM orders GROUP BY city ORDER BY order_count DESC;
''', conn)

plt.figure(figsize=(12, 7))
sns.barplot(x='order_count', y='city', data=df2, palette='Oranges_r')
plt.title("City-wise Order Volume Distribution", fontsize=18, fontweight='bold')
plt.xlabel("Total Orders Placed")
plt.ylabel("City")
plt.tight_layout()
plt.show()

### Chart 3: Delivery Time Heatmap (Hour of Day vs Day of Week)
Highlights hourly order logistics and peak delivery time bottlenecks.

In [ ]:
df3 = pd.read_sql_query('''
    SELECT 
        CASE strftime('%w', o.order_date)
            WHEN '0' THEN 'Sunday' WHEN '1' THEN 'Monday' WHEN '2' THEN 'Tuesday' 
            WHEN '3' THEN 'Wednesday' WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday' 
            WHEN '6' THEN 'Saturday'
        END AS day_of_week,
        CAST(strftime('%H', o.order_time) AS INTEGER) AS order_hour,
        AVG(dt.actual_delivery_minutes) AS avg_delivery_time
    FROM delivery_tracking dt JOIN orders o ON dt.order_id = o.order_id
    GROUP BY day_of_week, order_hour;
''', conn)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df3['day_of_week'] = pd.Categorical(df3['day_of_week'], categories=day_order, ordered=True)
pivot_df3 = df3.pivot(index='day_of_week', columns='order_hour', values='avg_delivery_time')

plt.figure(figsize=(14, 8))
sns.heatmap(pivot_df3, cmap='YlOrRd', annot=False, cbar_kws={'label': 'Avg Delivery Time (mins)'})
plt.title("Delivery Time Heatmap by Hour & Day of Week", fontsize=18, fontweight='bold')
plt.xlabel("Hour of Day (24h format)")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()

### Chart 4: RFM Customer Segments
Donut chart detailing proportions of Champions, Loyal, At Risk, and Lost customers.

In [ ]:
df4 = pd.read_sql_query('''
    WITH customer_rfm_raw AS (
        SELECT customer_id, (julianday('2024-01-01') - julianday(MAX(order_date))) AS recency,
            COUNT(order_id) AS frequency, SUM(total_amount) AS monetary
        FROM orders WHERE order_status = 'Delivered' GROUP BY customer_id
    ),
    rfm_scores AS (
        SELECT customer_id,
            CASE WHEN recency <= 45 THEN 4 WHEN recency <= 90 THEN 3 WHEN recency <= 180 THEN 2 ELSE 1 END AS r_score,
            CASE WHEN frequency >= 15 THEN 4 WHEN frequency >= 8 THEN 3 WHEN frequency >= 3 THEN 2 ELSE 1 END AS f_score,
            CASE WHEN monetary >= 10000 THEN 4 WHEN monetary >= 5000 THEN 3 WHEN monetary >= 1500 THEN 2 ELSE 1 END AS m_score
        FROM customer_rfm_raw
    )
    SELECT 
        CASE 
            WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Champions'
            WHEN r_score <= 2 AND f_score <= 2 AND m_score <= 2 THEN 'Lost'
            WHEN r_score <= 2 AND (f_score >= 3 OR m_score >= 3) THEN 'At Risk'
            ELSE 'Loyal'
        END AS customer_segment, COUNT(*) AS segment_count
    FROM rfm_scores GROUP BY customer_segment;
''', conn)

plt.figure(figsize=(10, 7))
wedges, texts, autotexts = plt.pie(df4['segment_count'], labels=df4['customer_segment'], autopct='%1.1f%%', 
                                  startangle=90, colors=['#FF6F61', '#4F9D69', '#FFD166', '#92A8D1'],
                                  textprops=dict(color='black', fontweight='bold'), pctdistance=0.75)
centre_circle = plt.Circle((0,0), 0.55, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title("RFM Customer Segmentation Share", fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

### Chart 5: Restaurant Performance Scatter (Revenue vs Rating)
Compares revenue output to ratings for individual restaurants.

In [ ]:
df5 = pd.read_sql_query('''
    SELECT r.name, r.city, r.rating, COUNT(o.order_id) AS order_count, SUM(o.total_amount) AS revenue 
    FROM orders o JOIN restaurants r ON o.restaurant_id = r.restaurant_id 
    WHERE o.order_status = 'Delivered' 
    GROUP BY r.restaurant_id, r.name, r.city, r.rating;
''', conn)

df5_sample = df5.sample(min(1000, len(df5)), random_state=42)
plt.figure(figsize=(12, 8))
sns.scatterplot(x='rating', y='revenue', size='order_count', hue='city', sizes=(20, 400), alpha=0.6, 
                data=df5_sample, palette='tab10')
plt.title("Restaurant Performance Analysis (Revenue vs Rating)", fontsize=18, fontweight='bold')
plt.xlabel("Restaurant Rating")
plt.ylabel("Total Revenue Generated (INR)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='City / Vol')
plt.tight_layout()
plt.show()

### Chart 6: On-time Delivery Rate by City
Plots city logistics performance against the benchmark line.

In [ ]:
df6 = pd.read_sql_query('''
    SELECT o.city, AVG(dt.was_on_time) * 100.0 AS on_time_rate 
    FROM delivery_tracking dt JOIN orders o ON dt.order_id = o.order_id 
    GROUP BY o.city ORDER BY on_time_rate DESC;
''', conn)

plt.figure(figsize=(12, 7))
sns.barplot(x='on_time_rate', y='city', data=df6, palette='Blues_r')
plt.axvline(x=75, color='red', linestyle='--', linewidth=2, label='Target Benchmark (75%)')
plt.title("On-Time Delivery Rate by City", fontsize=18, fontweight='bold')
plt.xlabel("On-Time Percentage (%)")
plt.ylabel("City")
plt.legend(loc='lower left')
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

### Chart 7: Cuisine Market Share by City
Stacked bar detailing cuisine market splits across top 5 cities.

In [ ]:
df7 = pd.read_sql_query('''
    WITH top_cities AS (SELECT city, COUNT(order_id) AS vol FROM orders GROUP BY city ORDER BY vol DESC LIMIT 5),
    cuisine_shares AS (
        SELECT o.city,
            CASE WHEN instr(r.food_type, ',') > 0 THEN substr(r.food_type, 1, instr(r.food_type, ',') - 1) ELSE r.food_type END AS cuisine_type,
            COUNT(o.order_id) AS orders_count
        FROM orders o JOIN restaurants r ON o.restaurant_id = r.restaurant_id
        WHERE o.city IN (SELECT city FROM top_cities) AND o.order_status = 'Delivered'
        GROUP BY o.city, cuisine_type
    )
    SELECT city, cuisine_type, orders_count FROM cuisine_shares;
''', conn)

pivot_df7 = df7.pivot(index='city', columns='cuisine_type', values='orders_count').fillna(0)
top_cuisines = df7.groupby('cuisine_type')['orders_count'].sum().nlargest(5).index
main_cuisines = pivot_df7[top_cuisines].copy()
main_cuisines['Others'] = pivot_df7.drop(columns=top_cuisines).sum(axis=1)
main_cuisines_pct = main_cuisines.div(main_cuisines.sum(axis=1), axis=0) * 100.0

main_cuisines_pct.plot(kind='bar', stacked=True, figsize=(13, 8), color=colors)
plt.title("Cuisine Market Share across Top 5 Cities", fontsize=18, fontweight='bold')
plt.xlabel("City")
plt.ylabel("Market Share Percentage (%)")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Cuisines')
plt.tight_layout()
plt.show()

### Chart 8: Delivery Partner Performance Quadrants
Maps partners by on-time delivery rates vs average review ratings.

In [ ]:
df8 = pd.read_sql_query('''
    SELECT dp.partner_id, AVG(dt.was_on_time) * 100.0 AS on_time_rate, AVG(r.delivery_rating) AS avg_rating 
    FROM delivery_partners dp JOIN delivery_tracking dt ON dp.partner_id = dt.partner_id 
    LEFT JOIN reviews r ON dt.order_id = r.order_id GROUP BY dp.partner_id;
''', conn)
df8 = df8.dropna()

plt.figure(figsize=(12, 8))
sns.scatterplot(x='on_time_rate', y='avg_rating', alpha=0.5, color='#4A90E2', data=df8)
avg_on_time = df8['on_time_rate'].mean()
avg_rating = df8['avg_rating'].mean()

plt.axvline(x=avg_on_time, color='red', linestyle='--', linewidth=1.5, label=f'Avg On-Time ({avg_on_time:.1f}%)')
plt.axhline(y=avg_rating, color='red', linestyle='--', linewidth=1.5, label=f'Avg Rating ({avg_rating:.2f})')
plt.title("Delivery Partner Performance Quadrants", fontsize=18, fontweight='bold')
plt.xlabel("On-Time Delivery Rate (%)")
plt.ylabel("Average Delivery Rating")
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
conn.close()